![LVMM logo](https://lvmm.mx/wp-content/uploads/2019/05/LVMM_logo_500-90x90.png)

# Manual de uso del clúster LVMM

> Este documento ofrece una guía práctica para usuarios del Departamento de Modelación de Nanomateriales, CNyN-UNAM.
>
> Recomendación: revise siempre la configuración actual del clúster (`sinfo`, `module avail`, particiones disponibles y versiones instaladas) antes de enviar trabajos de cálculo.

## Resumen rápido

1. Acceder al clúster mediante SSH.
2. Transferir archivos con `sftp` o `scp`.
3. Cargar el módulo correcto con `module available` / `ml av`.
4. Ejecutar trabajos con `srun` o, mejor aún, `sbatch`.
5. Revisar el estado con `squeue` y cancelar si hace falta con `scancel`.

## Índice
- [1. ¿Cómo acceder al cluster LVMM?](#ssh)
- [2. ¿Cómo transferir información al clúster LVMM?](#sftp)
- [3. ¿Cómo utilizar los programas instalados?](#module)
- [4. ¿Cómo correr una aplicación en el LVMM?](#slurm)
  - [4.1 sinfo](#sinfo)
  - [4.2 srun](#srun)
  - [4.3 sbatch](#sbatch)
  - [4.4 squeue](#squeue)
  - [4.5 scancel](#scancel)
  - [4.6 salloc](#salloc)
- [5. Referencias](#references)

> En cada sección se sigue el mismo patrón: objetivo, comando, resultado esperado y advertencias importantes.

<a name="ssh"></a>
## 1. ¿Cómo acceder al cluster LVMM?

La conexión al clúster computacional del [Departamento de Modelación de Nanomateriales](https://www.cnyn.unam.mx/?page_id=659) del [CNyN-UNAM](https://www.cnyn.unam.mx) se realiza mediante SSH.

Dentro de la red del CNyN, la conexión se realiza con el siguiente comando:

```bash
$ ssh usuario@192.168.100.237
```

donde `usuario` es el nombre del usuario con el que se desea acceder al clúster.

Una vez establecida la conexión, el servidor solicitará la contraseña del usuario:

```bash
By accessing this system, you consent to the following conditions:
- This system is for authorized use only.
- Any or all uses of this system and all files on this system may be monitored.
- Communications using, or data stored on, this system are not private.

usuario@192.168.100.237's password:
```

Si la autenticaci��n es exitosa, se obtiene el símbolo del sistema del clúster:

```bash
usuario@lvmm:~$
```

> Nota: reemplace `usuario` por el nombre real de su cuenta en el sistema. Los nombres de host y direcciones IP pueden variar según la infraestructura del clúster.

<a name="sftp"></a>
## 2. ¿Cómo transferir información al clúster LVMM?

Para transferir información entre su equipo y el clúster, se utiliza el mismo protocolo SSH mediante las herramientas `sftp` y `scp`.

### sftp
`sftp` es la implementación del protocolo FTP sobre SSH. La forma más básica de utilizarlo es la siguiente:

```bash
$ sftp usuario@192.168.100.237
```

Una vez dentro del entorno `sftp`, puede listar, subir o descargar archivos con comandos semejantes a los de un shell estándar.

### scp
`scp` es una implementación del comando `cp` sobre SSH. Su sintaxis general es:

> scp [opciones] <origen> <destino>

Si el origen o el destino son remotos, la ruta deberá indicarse con el formato:

> usuario@192.168.100.237:<ruta/al/archivo_o_directorio>

Si se trata de un directorio, es importante incluir la opción `-r` para copiarlo de forma recursiva. Por ejemplo, para copiar un directorio remoto a su computadora local:

```bash
localhost:~/resultados $ scp -r usuario@192.168.100.237:/tmpu/grupo/usuario/datos .
```

Esto copia el directorio remoto `/tmpu/grupo/usuario/datos` del servidor `192.168.100.237` al directorio actual `.` en su equipo local.

También puede usar `scp` para transferir archivos del equipo local al clúster:

```bash
$ scp archivo.txt usuario@192.168.100.237:/tmpu/grupo/usuario/
```

> Recomendación: verifique siempre la ruta de destino y asegúrese de tener permisos de escritura en el directorio remoto.

<a name="module"></a>
# 3. ¿Cómo utilizar los programas instalados en el clúster LVMM?

Para utilizar los programas ya instalados en el clúster LVMM, es necesario cargar el módulo adecuado con [Environment Modules](https://modules.sourceforge.net/).

### Ver la lista de módulos disponibles
Para consultar la lista de aplicaciones, bibliotecas y utilidades disponibles, se puede ejecutar:

```bash
$ module available
```

o en su forma abreviada:

```bash
$ ml av
```

```bash
----------------------------------- /share/modules/Applications ------------------------------------
abinit/10.0.7.1-intel    gromacs/2024.1-gnu-mkl     vasp/6.4.3-intel           
amber/20-Update12        irrep/1.1                  vasp/6.4.3-intel-vtst         
autodock/4.2.6-intel     lammps/2Aug2023-intel      vasp/6.4.3-MD-intel           
boltztrap/20.7.1         namd/2.14-intel            vasp/6.4.3-nvhpc-gpu            
cp2k/2024.1-gnu-mkl      py4vasp/0.5.1              wannier90/2.1.0-intel
critic2/1.1.git-intel    qe/7.1-intel               wannier90/3.1.0-intel         
crystal17/1.0.2-intel    siesta/honpas-4.1.5-intel  wannier90/3.1.0-serial-intel  
dftb+/24.1-intel         tinker/8.10.2-intel        wien2k/23.2-intel             
gromacs/2024.1-gnu-cuda  

Key:
default-version  modulepath  
```

### Cargar un módulo
Si se desea utilizar, por ejemplo, VASP para GPU, se debe cargar el módulo correspondiente:

```bash
$ ml load vasp/6.4.3-nvhpc-gpu
```

La salida típica puede verse así:

```bash
Loading vasp version 6.4.3
Loading mkl version 2023.0.0
Loading tbb version 2021.8.0
Loading compiler-rt version 2023.0.0
Loading fftw version 3.3.10
Loading hdf5 version 1.14.3

Loading vasp/6.4.3-nvhpc-gpu
  Loading requirement: nvhpc/24.5 tbb/latest compiler-rt/latest mkl/latest fftw/3.3.10-nvhpc
    hdf5/1.14.3-nvhpc
```

Los mensajes de salida indican que también se han cargado las dependencias requeridas para que funcione correctamente la aplicación.

### Importante: evitar ejecutar en el nodo maestro
Una vez cargado el módulo, la ejecución directa de programas en el nodo de acceso o `login node` no es recomendada para cálculos intensivos. En cambio, se debe usar la cola de trabajos de SLURM.

Por ejemplo, si se ejecuta algo de forma directa, se puede ver algo como esto:

```bash
$ mpirun -n 4 vasp_ncl
 running    4 mpi-ranks, with    2 threads/rank, on    1 nodes
 distrk:  each k-point on    4 cores,    1 groups
 distr:  one band on    1 cores,    4 groups
 vasp.6.4.3 05Apr23 (build Jul  1 2024 01:14:50) complex
```

> Esto puede servir para pruebas rápidas o depuración, pero no es la forma recomendada para correr trabajos de producción. Para cálculos reales, use `srun` o `sbatch` dentro de SLURM.

<a name="slurm"></a>
# 4. ¿Cómo correr una aplicación en el LVMM?

El clúster HPC LVMM cuenta con el sistema de administración de recursos [SLURM](https://slurm.schedmd.com) (Simple Linux Utility for Resource Management).

SLURM permite reservar recursos, programar trabajos y gestionarlos de forma ordenada. Las instrucciones básicas más importantes son `sinfo`, `srun`, `sbatch`, `squeue`, `scancel` y `salloc`.

> Patrón recomendado en esta sección: 1) revisar particiones, 2) reservar recursos, 3) enviar el trabajo, 4) monitorear su estado y 5) cancelar solo si es necesario.

<a name="sinfo"></a>
## 4.1 sinfo

El comando `sinfo` muestra información sobre el estado del clúster y sus particiones.

```bash
$ sinfo
```

```bash
PARTITION AVAIL  TIMELIMIT  NODES  STATE NODELIST
q_gpu        up   infinite      2   idle compute-0-[0,5]
q_mem        up   infinite      1    mix compute-0-1
q_cpu*       up   infinite      3    mix compute-0-[1-2,4]
q_cpu*       up   infinite      1  alloc compute-0-3
```

Este comando muestra las distintas particiones del clúster; en este ejemplo existen tres: `q_gpu`, `q_mem` y `q_cpu`. La palabra `idle` indica que un nodo está disponible; `mix` indica que hay recursos mezclados; `alloc` indica que está ocupado.

También puede obtenerse la misma información enfocada en nodos, usando la opción `--Node` o `-N`:

```bash
$ sinfo --Node
```

```bash
NODELIST     NODES PARTITION STATE
compute-0-0      1     q_gpu idle
compute-0-1      1    q_cpu* mix
compute-0-1      1     q_mem mix
compute-0-2      1    q_cpu* mix
compute-0-3      1    q_cpu* alloc
compute-0-4      1    q_cpu* mix
compute-0-5      1     q_gpu idle
```

> Importante: los nombres de particiones, nodos y capacidades pueden cambiar según la configuración del sistema. Consulte siempre `sinfo` antes de enviar un trabajo.

<a name="srun"></a>
## 4.2 srun

`srun` se usa para ejecutar un programa en una reserva de recursos de forma interactiva o inmediata. Por ejemplo, para correr VASP en la partición `q_mem` con 16 tareas:

```bash
$ srun --partition q_mem --ntasks 16 vasp_ncl
```

Salida típica:

```bash
 running   16 mpi-ranks, on    1 nodes
 distrk:  each k-point on   16 cores,    1 groups
 distr:  one band on    4 cores,    4 groups
 vasp.6.4.3 19Mar24 (build Mar 30 2025 15:09:16) complex
```

Con esto se ejecuta `vasp_ncl` en la partición `q_mem` con 16 tareas. El inconveniente de esta forma es que el usuario debe esperar a que los recursos estén disponibles y, mientras el trabajo está corriendo, la sesión puede seguir ligada a la terminal.

Para trabajos largos o repetitivos, la opción recomendada es `sbatch`.

<a name="sbatch"></a>
## 4.3 sbatch

La forma idónea para enviar un trabajo a la cola de ejecución es mediante `sbatch`, creando primero un script de trabajo.

Ejemplo mínimo (`vasp.job`):

```shell
#!/usr/bin/env bash
srun vasp_ncl
```

Una vez creado el script, se envía con `sbatch` de la siguiente forma:

```bash
$ sbatch --ntasks=16 --nodes=1 --partition=q_cpu vasp.job
```

Salida típica:

```bash
Submitted batch job 176
```

El número `176` es el identificador del trabajo (`JOBID`).

También se pueden incluir las opciones de `sbatch` dentro del script, junto con otras configuraciones importantes:

`vasp.job:`
```shell
#!/bin/bash
#SBATCH --job-name=VASP_test          # Nombre para identificar el trabajo (-J)
#SBATCH --nodes=1                    # Utilizar sólo un nodo (-N)
#SBATCH --ntasks=16                  # Ejecutar 16 tareas (-n)
#SBATCH --cpus-per-task=1            # CPUs por tarea (-c)
#SBATCH --ntasks-per-node=16         # Tareas por nodo
#SBATCH --output=vasp_test-%j.out    # Archivo para la salida estándar (-o)
#SBATCH --error=vasp_test-%j.err     # Archivo para los errores (-e)
# %j se refiere al ID asignado al enviar el script
#SBATCH --partition=q_cpu            # Partición en la que ejecutar el script (-p)

. /etc/profile.d/modules.sh
ml vasp/6.4.3-intel

srun vasp_ncl
```

Para enviar este último script, basta con invocar `sbatch` con el nombre del archivo:

```bash
$ sbatch vasp.job
```

```bash
Submitted batch job 177
```

> Recomendación: verifique la partición correcta, el número de tareas y la versión del módulo antes de enviar trabajos de producción. Los valores de ejemplo son ilustrativos y pueden cambiar.

<a name="squeue"></a>
## 4.4 squeue

Para consultar el estado de los trabajos enviados a SLURM, se utiliza `squeue`.

```bash
$ squeue
```

```bash
             JOBID PARTITION     NAME     USER ST       TIME  NODES NODELIST(REASON)
             164       q_cpu  model_4   jperez  R 16-10:19:13     1 compute-0-3
             174       q_cpu    model   jperez  R 4-07:57:35      1 compute-0-2
             173       q_cpu     VASP vgalingo  R 2-12-45:11      1 compute-0-1
             175       q_cpu     Test  wmedina  R   22:59:14      1 compute-0-2
             171       q_cpu      aex  wmedina  R 1-20:57:02      1 compute-0-3
             172       q_cpu   10-35s   clfdez  R    7:05:41      1 compute-0-1
             177       q_cpu VASP_tes    suser  R      16:32      1 compute-0-1
             178       q_cpu model_dm  emarmol  R    6:06:50      1 compute-0-3
             182       q_cpu model_fm vgalindo  R    5:47:16      1 compute-0-4
             184       q_cpu      igc  ncampos  R      56:32      1 compute-0-1
             186       q_mem Na2SO2_d    jvzmg  R    7:05:41      1 compute-0-1
```

La columna `ST` indica el estado del trabajo. En este ejemplo, `R` significa que el trabajo está corriendo.

Para ver solo los trabajos del usuario actual:

```bash
$ squeue --user $USER
```

```bash
             JOBID PARTITION     NAME     USER ST       TIME  NODES NODELIST(REASON)
             177       q_cpu VASP_tes    suser  R      18:22      1 compute-0-1
```

Otros estados comunes incluyen `PD` (pendiente), `CG` (cancelling), `CA` (cancelado) y `CD` (completado).

<a name="scancel"></a>
## 4.5 scancel

Se puede cancelar o retirar un trabajo de la cola con el comando `scancel`:

```bash
$ scancel 177
```

Para verificar que el trabajo quedó cancelado:

```bash
$ squeue -u $USER
```

```bash
             JOBID PARTITION     NAME     USER ST       TIME  NODES NODELIST(REASON)
             177         cpu VASP_tes    suser  CA      18:22      1 compute-0-1
```

La columna `ST` cambia a `CA`, que indica que el trabajo está cancelado. También es posible cancelar por nombre del trabajo usando la opción `--name`:

```bash
$ scancel --name VASP_test
```

> Consejo: use `scancel` únicamente cuando realmente necesite detener un trabajo; asegúrese de que el job ID o nombre coinciden con el trabajo correcto.

<a name="salloc"></a>
## 4.6 salloc

También es posible ejecutar programas de forma interactiva usando `salloc`, el cual reserva recursos y entrega un shell interactivo en el nodo asignado.

Ejemplo:

```bash
$ salloc -p q_cpu -N 1 -n 16 -J VASP_test -t 60
```

Salida típica:

```bash
salloc: Granted job allocation 178
$
```

Una vez asignados los recursos, el usuario puede ejecutar comandos dentro del shell interactivo. Cuando termine la sesión, puede salir con cualquiera de los comandos habituales:

```bash
$ exit
```
o
```bash
$ logout
```

También puede utilizar la combinación rápida `Ctrl + D` para cerrar la sesión.

> `salloc` es útil para pruebas interactivas y depuración, pero para trabajos reproducibles y de larga ejecución es preferible usar `sbatch`.

<a name="references"></a>
---
# 5. Referencias

- The SLURM website: https://slurm.schedmd.com/
- The SLURM documentation: https://slurm.schedmd.com/documentation.html
- The SLURM user community: https://groups.google.com/g/slurm-users
- Environment Modules: https://modules.sourceforge.net/

Aldo Rodríguez Guerrero, 2024-2026